### Clustering prediction

In [94]:
import os
print(os.getcwd())


c:\Users\DELL\AppData\Local\Programs\Microsoft VS Code


In [95]:
import joblib

# ✅ Correct way: Load using the actual full paths
kmeans = joblib.load(r"D:\Downloads\kmeans_model.pkl")
scaler = joblib.load(r"D:\Downloads\scaler.pkl")

print("✅ Model and scaler loaded successfully!")


✅ Model and scaler loaded successfully!


In [ ]:
import pandas as pd

# ✅ Load your test dataset
data = pd.read_csv(r"D:\Downloads\customer_data_with_products.csv")

# ✅ Select the same features you used during training
features = ['Recency', 'Avg_Order_Value', 'Customer_Lifetime_Days', 'Purchase_Rate', 'Total_Items_Sold']

# ✅ Prepare and scale the data
X = data[features]
X_scaled = scaler.transform(X)

# ✅ Predict cluster labels
data["Predicted_Cluster"] = kmeans.predict(X_scaled)

# ✅ Map cluster labels to descriptive names
cluster_mapping = {
    0: "Dormant/Churned",
    1: "Loyal/Engaged",
    2: "New/Recent but Inactive",
    3: "High-Engagement/Recent High-Value"
}

data["Cluster_Description"] = data["Predicted_Cluster"].map(cluster_mapping)

# ✅ Save the results
data.to_csv(r"D:\Downloads\cluster_results.csv", index=False)

import pandas as pd
import numpy as np

# -------------------------------
# 1️⃣ Load your test dataset
# -------------------------------
data_path = r"D:\Downloads\customer_data_with_products.csv"
df = pd.read_csv(data_path)

print("✅ Data loaded successfully. Shape:", df.shape)
print(df.head())

# -------------------------------
# 2️⃣ Select the same features used during training
# -------------------------------
features = ['Recency', 'Avg_Order_Value', 'Customer_Lifetime_Days', 'Purchase_Rate', 'Total_Items_Sold']

# Check for missing features
missing = [f for f in features if f not in df.columns]
if missing:
    raise ValueError(f"Missing feature columns in data: {missing}")

# -------------------------------
# 3️⃣ Prepare and scale the data
# -------------------------------
X = df[features]
X_scaled = scaler.transform(X)  # assumes your trained scaler is already loaded in memory

# -------------------------------
# 4️⃣ Predict cluster labels
# -------------------------------
df["Predicted_Cluster"] = kmeans.predict(X_scaled)

# -------------------------------
# 5️⃣ Map cluster labels to descriptive names
# -------------------------------
cluster_mapping = {
    0: "Dormant/Churned",
    1: "Loyal/Engaged",
    2: "New/Recent but Inactive",
    3: "High-Engagement/Recent High-Value"
}
df["Cluster_Description"] = df["Predicted_Cluster"].map(cluster_mapping)

print("\n🔮 Cluster Prediction Summary:")
print(df[["Predicted_Cluster", "Cluster_Description"]].head())

# -------------------------------
# 6️⃣ Top Products per Cluster
# -------------------------------
print("\n📦 Analyzing top products by cluster...")

product_cols = [col for col in df.columns if col.startswith('Product_')]

if not product_cols:
    print("⚠️ No product columns found (columns starting with 'Product_').")
    top_products_by_cluster = None
else:
    # Combine all product columns into one Series per customer
    df_melted = df.melt(
        id_vars=['Cluster_Description'],
        value_vars=product_cols,
        var_name='Product_Column',
        value_name='Product'
    ).dropna(subset=['Product'])

    # Group by cluster and product to count occurrences
    product_counts = (
        df_melted.groupby(['Cluster_Description', 'Product'])
        .size()
        .reset_index(name='Count')
        .sort_values(['Cluster_Description', 'Count'], ascending=[True, False])
    )

    # Separate top products per cluster
    top_products_by_cluster = {}
    for cluster_name in df['Cluster_Description'].unique():
        top_products = (
            product_counts[product_counts['Cluster_Description'] == cluster_name]
            .sort_values('Count', ascending=False)
        )
        top_products_by_cluster[cluster_name] = top_products

    print("\n🏆 Top Products per Cluster:")
    for cluster_name, table in top_products_by_cluster.items():
        print(f"\n🟩 {cluster_name}:")
        print(table.head(10))

# -------------------------------
# 7️⃣ Save results to Excel
# -------------------------------
output_file = r"D:\Downloads\cluster_results_with_products.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Cluster_Predictions', index=False)
    if top_products_by_cluster:
        for cluster_name, table in top_products_by_cluster.items():
            sheet_name = cluster_name.replace("/", "_").replace(" ", "_")[:31]  # Excel sheet name limit
            table.to_excel(writer, sheet_name=f"Top_Products_{sheet_name}", index=False)

print(f"\n💾 Results (clusters + top products) saved successfully to '{output_file}'")


print("✅ Predictions completed and saved to D:\\Downloads\\cluster_results.csv")
data


✅ Data loaded successfully. Shape: (1000, 10)
     Customer_ID  Gender  Age   Region  Recency  Avg_Order_Value  \
0  CUS_TEST_0001    Male   64     East      176           122.29   
1  CUS_TEST_0002  Female   29  Central      116           277.97   
2  CUS_TEST_0003  Female   33    South      204           338.63   
3  CUS_TEST_0004  Female   41     East      321           353.43   
4  CUS_TEST_0005    Male   36     West       31           109.97   

   Customer_Lifetime_Days  Purchase_Rate  Total_Items_Sold Product_Purchased  
0                     642          0.014                80        Headphones  
1                    1209          0.036                36        Smartphone  
2                    1393          0.002                84           Speaker  
3                     154          0.090                37           Printer  
4                     900          0.018                86           Monitor  

🔮 Cluster Prediction Summary:
   Predicted_Cluster                Clus

c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")



💾 Results (clusters + top products) saved successfully to 'D:\Downloads\cluster_results_with_products.xlsx'
✅ Predictions completed and saved to D:\Downloads\cluster_results.csv


,Customer_ID,Gender,Age,Region,Recency,Avg_Order_Value,Customer_Lifetime_Days,Purchase_Rate,Total_Items_Sold,Product_Purchased,Predicted_Cluster,Cluster_Description
0,CUS_TEST_0001,Male,64,East,176,122.29,642,0.014,80,Headphones,1,Loyal/Engaged
1,CUS_TEST_0002,Female,29,Central,116,277.97,1209,0.036,36,Smartphone,1,Loyal/Engaged
2,CUS_TEST_0003,Female,33,South,204,338.63,1393,0.002,84,Speaker,1,Loyal/Engaged
3,CUS_TEST_0004,Female,41,East,321,353.43,154,0.090,37,Printer,3,High-Engagement/Recent High-Value
4,CUS_TEST_0005,Male,36,West,31,109.97,900,0.018,86,Monitor,1,Loyal/Engaged
...,...,...,...,...,...,...,...,...,...,...,...,...
995,CUS_TEST_0996,Male,30,East,271,251.37,1591,0.127,33,Gaming Console,3,High-Engagement/Recent High-Value
996,CUS_TEST_0997,Female,66,East,132,320.37,1436,0.106,40,Headphones,3,High-Engagement/Recent High-Value
997,CUS_TEST_0998,Male,19,North,93,404.71,1168,0.117,32,Laptop,3,High-Engagement/Recent High-Value
998,CUS_TEST_0999,Female,22,North,73,204.96,923,0.137,69,Television,1,Loyal/Engaged


## Retention prediction

In [ ]:
import pandas as pd
import numpy as np
import joblib

# -------------------------------
# 1️⃣ Load the trained model
# -------------------------------
model_path = r"D:\Downloads\customer_retention_model.pkl"
loaded_model = joblib.load(model_path)
print("✅ Model loaded successfully.")

# -------------------------------
# 2️⃣ Load your actual data
# -------------------------------
data_path = r"D:\Downloads\dummy_customer_data.csv"
df = pd.read_csv(data_path)

print("\n📊 Data loaded successfully. Shape:", df.shape)
print(df.head())

# -------------------------------
# 3️⃣ Select or verify feature columns used for the model
# -------------------------------
features = [
    'Recency_y', 'Frequency', 'Monetary', 'Customer_Lifetime_Days', 'Purchase_Rate',
    'Total_Items_Sold', 'Unique_Products_Count_y', 'Avg_Items_Per_Order_y',
    'Avg_Revenue_Per_Order_y', 'Avg_Net_Sales_Per_Order_y'
]

missing_features = [col for col in features if col not in df.columns]
if missing_features:
    raise ValueError(f" Missing feature columns in data: {missing_features}")

X = df[features].copy()

# -------------------------------
# 4️⃣ Handle missing or infinite values
# -------------------------------
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(X.mean(), inplace=True)

# -------------------------------
# 5️⃣ Make predictions
# -------------------------------
predictions = loaded_model.predict(X)
df['Predicted_Retention'] = predictions
df['Retention_Status'] = df['Predicted_Retention'].map({1: 'Returning', 0: 'Not Returning'})

print("\n🔮 Prediction Summary:")
print(df[['Predicted_Retention', 'Retention_Status']].head())

# -------------------------------
# 6️⃣ Find Top Products per Retention Class
# -------------------------------
print("\n📦 Analyzing top products by retention class...")

product_cols = [col for col in df.columns if col.startswith('Product_')]

if not product_cols:
    print("⚠️ No product columns found (columns starting with 'Product_').")
    top_products_by_class = None
else:
    # Combine all product columns into one Series per customer
    df_melted = df.melt(
        id_vars=['Retention_Status'],
        value_vars=product_cols,
        var_name='Product_Column',
        value_name='Product'
    ).dropna(subset=['Product'])

    # Group by retention status and product to count occurrences
    product_counts = (
        df_melted.groupby(['Retention_Status', 'Product'])
        .size()
        .reset_index(name='Count')
        .sort_values(['Retention_Status', 'Count'], ascending=[True, False])
    )

    # Separate top products per class
    top_products_by_class = {}
    for status in df['Retention_Status'].unique():
        top_products = (
            product_counts[product_counts['Retention_Status'] == status]
            .sort_values('Count', ascending=False)
        )
        top_products_by_class[status] = top_products

    print("\n🏆 Top Products per Retention Class:")
    for status, table in top_products_by_class.items():
        print(f"\n🟩 {status} Customers:")
        print(table.head(10))

# -------------------------------
# 7️⃣ Save results to Excel
# -------------------------------
output_file = r"C:\Users\DELL\Desktop\agroX\agroX\customer_retention_predictions.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Predictions', index=False)
    if top_products_by_class:
        for status, table in top_products_by_class.items():
            sheet_name = status.replace(" ", "_")
            table.to_excel(writer, sheet_name=f"Top_Products_{sheet_name}", index=False)

print(f"\n💾 Results saved successfully to '{output_file}'")


c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.6.1 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.6.1 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


✅ Model loaded successfully.

📊 Data loaded successfully. Shape: (100, 13)
     Customer_ID  Gender Product_Purchased  Recency_y  Frequency  Monetary  \
0  CUS_TEST_0001    Male            Camera        151          3    552951   
1  CUS_TEST_0002  Female            Laptop        298          8     52991   
2  CUS_TEST_0003  Female        Headphones         99         14    189751   
3  CUS_TEST_0004  Female        Headphones        263         18    367326   
4  CUS_TEST_0005    Male        Smartwatch        252         15    687337   

   Customer_Lifetime_Days  Purchase_Rate  Total_Items_Sold  \
0                     798          0.112               335   
1                    1142          0.065                30   
2                     276          0.010               247   
3                     495          0.185               106   
4                     544          0.184                51   

   Unique_Products_Count_y  Avg_Items_Per_Order_y  Avg_Revenue_Per_Order_y  \
0    